# 任务四 Retina 48 像素小块

单通道灰度对比度预处理，32/64/128 通道浅层 U-Net，逐元素 Dropout 0.2。以 48 像素小块均匀采样训练，采用 AdamW、BCE 与 Dice、EMA 和 D4。

训练与推理使用当前目录的云端配置，完整结果与执行状态见任务四 README 和研究索引。

参考来源：https://github.com/orobix/retina-unet/tree/f6ff63bb39409942fd3aaf6907dbce74ca143207


In [ ]:
"""Retinal vessel segmentation. All training runs on Kaggle GPU.
No external data or pretrained vessel models are used.
"""
# Source method: https://github.com/orobix/retina-unet/tree/f6ff63bb39409942fd3aaf6907dbce74ca143207
# MIT License
# 
# Copyright (c) 2016 Daniele Cortinovis, Orobix Srl
# 
# Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:
# 
# The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.
# 
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
# 
import os, sys, json, time, random, hashlib, platform, copy
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

CFG = dict(seed=20261031, folds=4, steps=7000, batch_size=64,
           patch_size=48, learning_rate=0.001, image_size=512, warm_start=False,
           eval_every=1400, channels=[32,64,128], architecture='Retina shallow patch UNet', preprocessing='minmax-gray CLAHE gamma1.2', dropout=.2)
OUT = Path('/kaggle/working') if Path('/kaggle').exists() else Path('outputs')
OUT.mkdir(exist_ok=True)
START = time.time()
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'GPU is required; enable GPU in Kaggle Notebook settings.'
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device('cuda')
print('CONFIG', json.dumps(CFG), flush=True)
print('GPU', torch.cuda.get_device_name(0), 'Torch', torch.__version__, flush=True)
candidates=[p.parent for p in Path('/kaggle/input').rglob('sample_submission.csv') if (p.parent/'train/image').is_dir()]
assert len(candidates)==1, candidates
ROOT=candidates[0]
assert (ROOT / 'train/image').is_dir(), ROOT
print('DATA ROOT', ROOT, flush=True)

# Match official segmentation_to_csv.py: 512x512 bilinear, then >0.
def metric_mask(path):
    return np.asarray(Image.open(path).convert('L').resize((512,512), Image.Resampling.BILINEAR)) > 0

# Orobix retina-unet preprocessing, independently expressed.
# Global standardization cancels under the subsequent per-image min/max affine map.
# No train/validation/test population statistics are needed.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
GAMMA_TABLE=np.asarray([255*(v/255.)**(1/1.2) for v in range(256)],dtype=np.uint8)
def retina_channel(rgb):
    gray=rgb.astype(np.float64)@np.array([.299,.587,.114])
    gray=(255*(gray-gray.min())/max(float(gray.max()-gray.min()),1e-8)).clip(0,255).astype(np.uint8)
    return cv2.LUT(clahe.apply(gray),GAMMA_TABLE)

def load_image(path):
    rgb=np.asarray(Image.open(path).convert('RGB').resize((512,512),Image.Resampling.BILINEAR))
    aux=retina_channel(rgb)
    return aux[:,:,None]

train_paths = sorted((ROOT/'train/image').glob('*.jpg'), key=lambda p:int(p.stem))
test_paths = sorted((ROOT/'test/image').glob('*.jpg'), key=lambda p:int(p.stem))
assert len(train_paths)==20 and len(test_paths)==20
images = [load_image(p) for p in train_paths]
masks = [metric_mask(ROOT/'train/label'/p.name).astype(np.uint8) for p in train_paths]
test_images = [load_image(p) for p in test_paths]
print('Training foreground fractions', [round(float(m.mean()),4) for m in masks], flush=True)
manifest = {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in train_paths + [ROOT/'train/label'/p.name for p in train_paths] + test_paths}
(OUT/'data_manifest.json').write_text(json.dumps(manifest, indent=2))

class Patches(Dataset):
    def __init__(self,indices):self.indices=list(map(int,indices))
    def __len__(self):return CFG['steps']*CFG['batch_size']
    def __getitem__(self,index):
        i=random.choice(self.indices);s=CFG['patch_size']
        # Uniform crops include retinal boundary and outside-FOV background.
        y=random.randint(0,512-s);x=random.randint(0,512-s)
        im=images[i][y:y+s,x:x+s].copy();mask=masks[i][y:y+s,x:x+s].copy()
        k=random.randrange(4);im=np.rot90(im,k);mask=np.rot90(mask,k)
        if random.random()<.5:im=im[:,::-1];mask=mask[:,::-1]
        im=im.astype(np.float32)/255.
        if random.random()<.5:im=np.clip(im*random.uniform(.9,1.1)+random.uniform(-.03,.03),0,1)
        return torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1)*2-1)),torch.from_numpy(mask.copy()[None].astype(np.float32))

class RetinaBlock(nn.Module):
    def __init__(self,a,b):
        super().__init__();self.layers=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.ReLU(),nn.Dropout(.2),nn.Conv2d(b,b,3,padding=1),nn.ReLU())
    def forward(self,x):return self.layers(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__();self.enc=nn.ModuleList([RetinaBlock(1,32),RetinaBlock(32,64),RetinaBlock(64,128)])
        self.dec=nn.ModuleList([RetinaBlock(192,64),RetinaBlock(96,32)]);self.head=nn.Conv2d(32,1,1)
    def forward(self,x):
        skips=[]
        for i,block in enumerate(self.enc):
            if i:x=F.max_pool2d(x,2)
            x=block(x);skips.append(x)
        for block,skip in zip(self.dec,reversed(skips[:-1])):
            x=F.interpolate(x,size=skip.shape[-2:],mode='nearest');x=block(torch.cat([skip,x],1))
        return self.head(x)

def loss_fn(logits, targets):
    bce=F.binary_cross_entropy_with_logits(logits,targets)
    p=logits.float().sigmoid()
    dice=(2*(p*targets).sum((1,2,3))+1)/(p.sum((1,2,3))+targets.sum((1,2,3))+1)
    return .8*bce+.2*(1-dice.mean())

@torch.inference_mode()
def predict(model,ims,tta=True):
    model.eval(); result=[]
    for im in ims:
        x=torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))).float().to(device)[None]/127.5-1
        total=torch.zeros((1,1,512,512),device=device)
        variants=[(k,flip) for k in range(4) for flip in (False,True)] if tta else [(0,False)]
        for k,flip in variants:
            z=torch.rot90(x,k,(-2,-1))
            if flip:z=z.flip(-1)
            with torch.autocast('cuda',dtype=torch.float16): p=model(z).float().sigmoid()
            if flip:p=p.flip(-1)
            total+=torch.rot90(p,-k,(-2,-1))
        result.append((total/len(variants))[0,0].cpu().numpy())
    return np.stack(result)

@torch.inference_mode()
def predict_overlap(model,ims):
    model.eval();result=[];s=48;stride=12;pad=12;side=512+2*pad
    positions=list(range(0,side-s+1,stride))
    if positions[-1]!=side-s:positions.append(side-s)
    coords=[(y,x) for y in positions for x in positions]
    weight=torch.ones((s,s),device=device)
    for im in ims:
        x=torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))).float().to(device)[None]/127.5-1
        total=torch.zeros((512,512),device=device)
        # Four flips; denser than full-frame inference, hence only at final checkpoint.
        for axes in [(),(-2,),(-1,),(-2,-1)]:
            z=x.flip(axes) if axes else x
            z=F.pad(z,(pad,pad,pad,pad),mode='constant',value=-1)
            accum=torch.zeros((side,side),device=device);normal=torch.zeros_like(accum)
            for start in range(0,len(coords),128):
                batch_coords=coords[start:start+128]
                batch=torch.cat([z[:,:,y:y+s,x:x+s] for y,x in batch_coords])
                with torch.autocast('cuda',dtype=torch.float16):probs=model(batch).float().sigmoid()[:,0]
                for (y,xx),p in zip(batch_coords,probs):
                    accum[y:y+s,xx:xx+s]+=p*weight;normal[y:y+s,xx:xx+s]+=weight
            assert bool((normal>0).all())
            p=(accum/normal)[pad:pad+512,pad:pad+512]
            total+=p.flip(axes) if axes else p
        result.append((total/4).cpu().numpy())
    return np.stack(result)

THRESHOLDS=np.arange(.2,.751,.025)
def evaluate(probs,truth):
    truth=np.asarray(truth,dtype=bool)
    values=[]
    for t in THRESHOLDS:
        pred=probs>t
        inter=(pred&truth).sum((1,2)); den=pred.sum((1,2))+truth.sum((1,2))
        values.append(float(np.mean((2*inter+1e-8)/(den+1e-8))))
    k=int(np.argmax(values))
    return values[k],float(THRESHOLDS[k]),values

order=np.random.RandomState(CFG['seed']).permutation(20)
val_groups=np.array_split(order,CFG['folds'])
fold_split={str(f):[train_paths[i].stem for i in ids] for f,ids in enumerate(val_groups)}
(OUT/'folds.json').write_text(json.dumps(fold_split,indent=2))
overlap_oof=np.zeros((20,512,512),np.float32);overlap_test=np.zeros((20,512,512),np.float32)
oof=np.zeros((20,512,512),np.float32)
test_sum=np.zeros((20,512,512),np.float32)
summary=[]; history=[]
for fold,val_ids in enumerate(val_groups):
    seed=CFG['seed']+fold*97
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    train_ids=[i for i in range(20) if i not in val_ids]
    assert not set(train_ids).intersection(val_ids)
    print('FOLD',fold,'train',train_ids,'validation',val_ids.tolist(),flush=True)
    loader=DataLoader(Patches(train_ids),batch_size=CFG['batch_size'],num_workers=2,
                      pin_memory=True,drop_last=True,persistent_workers=True)
    model=UNet().to(device)
    print('MODEL PARAMETERS',sum(p.numel() for p in model.parameters()),flush=True)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=1e-4)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG['steps'],eta_min=3e-5)
    scaler=torch.amp.GradScaler('cuda')
    ema=copy.deepcopy(model).eval()
    for p in ema.parameters():p.requires_grad_(False)
    best=-1; running=0.; fold_start=time.time()
    initial=predict(model,[images[i] for i in val_ids])
    initial_score,_,_=evaluate(initial,[masks[i] for i in val_ids])
    print('INITIAL',fold,initial_score,flush=True)
    if CFG.get('warm_start'):
        best=initial_score
        torch.save(dict(model=model.state_dict(),config=CFG,fold=fold,val_ids=val_ids.tolist(),step=0,dice=best),OUT/f'fold{fold}.pt')
    for step,(x,y) in enumerate(loader,1):
        model.train(); x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16): loss=loss_fn(model(x),y)
        assert torch.isfinite(loss),'Nonfinite loss'
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),5.)
        scaler.step(optimizer); scaler.update(); schedule.step()
        with torch.no_grad():
            decay=min(.995,(1+step)/(10+step))
            for ep,p in zip(ema.parameters(),model.parameters()):ep.lerp_(p,1-decay)
            for eb,b in zip(ema.buffers(),model.buffers()):eb.copy_(b)
        running+=loss.item()
        if step%100==0:
            print(f'fold={fold} step={step}/{CFG["steps"]} loss={running/100:.5f} seconds={time.time()-fold_start:.0f}',flush=True)
            running=0.
        if step%CFG['eval_every']==0 or step==CFG['steps']:
            for variant,eval_model in [('raw',model),('ema',ema)]:
                vp=predict(eval_model,[images[i] for i in val_ids])
                score,threshold,_=evaluate(vp,[masks[i] for i in val_ids])
                row=dict(fold=fold,step=step,variant=variant,dice=score,threshold=threshold)
                history.append(row); print('VALIDATION',json.dumps(row),flush=True)
                if score>best:
                    best=score
                    torch.save(dict(model=eval_model.state_dict(),config=CFG,fold=fold,
                                    val_ids=val_ids.tolist(),step=step,dice=score,variant=variant),OUT/f'fold{fold}.pt')
            (OUT/'history.json').write_text(json.dumps(history,indent=2))
        if step>=CFG['steps']:break
    checkpoint=torch.load(OUT/f'fold{fold}.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(checkpoint['model'])
    vp=predict(model,[images[i] for i in val_ids]); oof[val_ids]=vp
    tp=predict(model,test_images); test_sum+=tp/CFG['folds']
    overlap_oof[val_ids]=predict_overlap(model,[images[i] for i in val_ids])
    overlap_test+=predict_overlap(model,test_images)/CFG['folds']
    np.savez_compressed(OUT/f'fold{fold}_probabilities.npz',val_ids=val_ids,oof=vp.astype(np.float16),test=tp.astype(np.float16))
    summary.append(dict(fold=fold,best_dice=best,best_step=checkpoint['step'],seconds=time.time()-fold_start))
    print('FOLD COMPLETE',json.dumps(summary[-1]),flush=True)
    del loader,optimizer,schedule,model,ema,checkpoint
    torch.cuda.empty_cache()

np.savez_compressed(OUT/'overlap_probabilities.npz',oof=overlap_oof.astype('float16'),test=overlap_test.astype('float16'),targets=np.asarray(masks),ids=np.arange(1,21))
overlap_score,overlap_threshold,_=evaluate(overlap_oof,masks)
print('OVERLAP RESULT',overlap_score,overlap_threshold,flush=True)
score,threshold,curve=evaluate(oof,masks)
np.savez_compressed(OUT/'probabilities.npz',oof=oof.astype(np.float16),test=test_sum.astype(np.float16),targets=np.asarray(masks),ids=np.arange(1,21))
report=dict(config=CFG,folds=summary,oof_dice=score,threshold=threshold,
            threshold_curve=[dict(threshold=float(t),dice=s) for t,s in zip(THRESHOLDS,curve)],
            seconds=time.time()-START,torch=torch.__version__,gpu=torch.cuda.get_device_name(0),
            notes='Orobix retina-unet-inspired grayscale preprocessing and 48px shallow patch UNet; modern AdamW/BCE+Dice/EMA/D4 augmentation; strict image-level folds. Full-frame and overlapping patch inference compared. White vessel labels; target encoding matches official 512px bilinear >0 conversion; image-level CV; no external data.')
(OUT/'metrics.json').write_text(json.dumps(report,indent=2))

# Official format is row-major (C order), 1-based run starts.
def encode_rle(mask):
    pixels=np.asarray(mask,dtype=np.uint8).ravel(order='C')
    padded=np.concatenate([[0],pixels,[0]])
    runs=np.flatnonzero(padded[1:]!=padded[:-1])+1
    runs[1::2]-=runs[::2]
    return ' '.join(map(str,runs))

def decode_rle(value):
    pairs=np.array(list(map(int,value.split())),dtype=int).reshape(-1,2)
    flat=np.zeros(512*512,np.uint8)
    for start,length in pairs: flat[start-1:start-1+length]=1
    return flat.reshape(512,512)

sample=pd.read_csv(ROOT/'sample_submission.csv',dtype={'Id':str}).fillna('')
by_id={p.stem:test_sum[i]>threshold for i,p in enumerate(test_paths)}
assert set(sample.Id)==set(by_id)
mask_dir=OUT/'predicted_masks'; mask_dir.mkdir(exist_ok=True)
encoded=[]
for ident in sample.Id:
    m=by_id[ident]; rle=encode_rle(m)
    assert np.array_equal(decode_rle(rle),m)
    assert 0.01<float(m.mean())<0.4,(ident,float(m.mean()))
    Image.fromarray(m.astype(np.uint8)*255).save(mask_dir/f'{ident}.png')
    encoded.append(rle)
sample['Predicted']=encoded
assert sample.shape==(20,2) and list(sample.columns)==['Id','Predicted'] and sample.Id.is_unique
sample.to_csv(OUT/'submission.csv',index=False)
# Verify original converter's exact operation against our encoding.
for ident,rle in zip(sample.Id,encoded):
    original=metric_mask(mask_dir/f'{ident}.png')
    assert np.array_equal(decode_rle(rle),original)
print('RESULT',json.dumps(report),flush=True)
print('SUBMISSION',OUT/'submission.csv','rows',len(sample),flush=True)
